In [3]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

import config


# 1. Load the webpage
url = "https://en.wikipedia.org/wiki/Cristiano_Ronaldo"

loader = WebBaseLoader(url)
raw_document = loader.load()


# 2. Split the document into smaller chunks
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_document)


# 3. Create embeddings and store them in FAISS
embeddings = OpenAIEmbeddings(api_key=config.api_key)

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

retriever = vectorstore.as_retriever()


# 4. Initialize the language model
llm = ChatOpenAI(
    api_key=config.api_key,
    model="gpt-4o-mini",
    temperature=0
)


# 5. Store conversation history
chat_history = []


# 6. Create the RAG chain
chain = (
    RunnablePassthrough.assign(
        context=lambda x: retriever.invoke(x["input"])
    )
    | ChatPromptTemplate.from_messages([
        (
            "system",
            "Answer the user's question based only on the following context:\n\n{context}"
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    | llm
    | StrOutputParser()
)


# 7. Create the chatbot function
def chat(question):
    response = chain.invoke({
        "input": question,
        "chat_history": chat_history
    })

    chat_history.append(
        HumanMessage(content=question)
    )

    chat_history.append(
        AIMessage(content=response)
    )

    return response


# Example
print(chat("When was Cristiano Ronaldo born?"))

Cristiano Ronaldo was born on 5 February 1985.


In [4]:
chat("how many champions league titles does he have?")

'Cristiano Ronaldo has won five UEFA Champions League titles.'

In [5]:
chat('who is Ronaldo\'s biggest rival?')

"Ronaldo's biggest rival is Lionel Messi."